## Update gamelist.xml files

Use the dataset to update gamelist.xml files

In [7]:
import pandas as pd
import os
from utils.update_gamelist import update_platform_gamelist, analyze_matching_breakdown

In [2]:
game_df = pd.read_pickle('cleaned_df.pkl')

In [3]:
# ============================================================
# TEST: Run full pipeline on one platform (NES)
# ============================================================

test_platform_abbrev = "nes"

# Run the complete pipeline for NES
result = update_platform_gamelist(
    platform_abbrev=test_platform_abbrev,
    game_df=game_df,
    lists_base_path="lists",
    threshold=85,
    verbose=True
)

# Extract results for analysis
if result['success']:
    matched_games = result['matched_games']
    audit = result['audit']
else:
    print(f"Pipeline failed: {result.get('error', 'Unknown error')}")
    matched_games = None
    audit = None


PLATFORM: nes → Nintendo Entertainment System
[1/4] Building inventory...
      Found 1184 games
[2/4] Matching to game_df metadata...
      Matched 1183/1184 games
[3/4] Generating updated gamelist.xml...
      Written to lists/nes/gamelist_updated.xml
[4/4] Generating audit report...

AUDIT REPORT: Nintendo Entertainment System
Total games in gamelist.xml: 1184
Matched to metadata: 1183 (99.9%)
Unmatched: 1
Low confidence matches (<90): 0

--- No Match Found (sample) ---
  DÃ©jÃ  Vu



In [4]:
# Analyze matching breakdown by type
if matched_games and audit:
    breakdown = analyze_matching_breakdown(matched_games, audit)


MATCHING STRATEGY BREAKDOWN

Matching Type Distribution:
  Filename (exact):      796 ( 67.2%)
  Filename (fuzzy):       86 (  7.3%)
  Name-based:            302 ( 25.5%)
  No match:                1 (  0.1%)
                        -----
  Total:                1184

Filename-based matches:  882 (74.5%)



In [ ]:
# ============================================================
# BULK PROCESSING: Run pipeline on multiple platforms
# ============================================================

# get folder names from lists
lists_folder = "lists"
platform_folders = [f for f in os.listdir(lists_folder) if os.path.isdir(os.path.join(lists_folder, f))]

# Select specific platforms to process (or use list(platform_mappings.keys()) for all)
platforms_to_process = ["nes", "snes", "genesis", "psx"]

print(f"\n{'='*70}")
print(f"PROCESSING {len(platforms_to_process)} PLATFORMS")
print(f"{'='*70}\n")

results_summary = []

for platform_abbrev in platform_folders:
    result = update_platform_gamelist(
        platform_abbrev=platform_abbrev,
        game_df=game_df,
        lists_base_path="lists",
        threshold=80,
        verbose=False  # Set to True to see details for each platform
    )
    
    if result['success']:
        results_summary.append({
            'Platform': result['platform_name'],
            'Total Games': result['inventory_count'],
            'Matched': result['matched_count'],
            'Match Rate (%)': f"{result['match_rate']*100:.1f}%",
            'Output': result['xml_output']
        })
        print(f"✓ {result['platform_name']:30s} | {result['matched_count']:4d}/{result['inventory_count']:4d} | {result['match_rate']*100:5.1f}%")
    else:
        print(f"✗ {platform_abbrev:30s} | Error: {result.get('error', 'Unknown')}")

# Display summary table
print(f"\n{'='*70}")
if results_summary:
    summary_df = pd.DataFrame(results_summary)
    print(summary_df.to_string(index=False))
else:
    print("No successful results to summarize")


PROCESSING 4 PLATFORMS

✓ Nintendo Entertainment System  | 1183/1184 |  99.9%
✓ Super Nintendo Entertainment System |  976/ 976 | 100.0%
✓ Sega Genesis                   | 1024/1024 | 100.0%
✓ Sony Playstation               |  265/ 265 | 100.0%

                           Platform  Total Games  Matched Match Rate (%)                             Output
      Nintendo Entertainment System         1184     1183          99.9%     lists/nes/gamelist_updated.xml
Super Nintendo Entertainment System          976      976         100.0%    lists/snes/gamelist_updated.xml
                       Sega Genesis         1024     1024         100.0% lists/genesis/gamelist_updated.xml
                   Sony Playstation          265      265         100.0%     lists/psx/gamelist_updated.xml
